## Session 9: Building Agents with the OpenAI Agents SDK

This notebook builds progressively more capable agents using the OpenAI Agents SDK:

1. **Hello Agent** — the simplest agent: no tools, just instructions
2. **Tool-using agent** — add a calculator, date tool, and FAISS search
3. **Inspect the trace** — see exactly what the agent did step by step
4. **Orchestrator-workers** — one agent delegates to specialists
5. **Handoffs** — transfer control permanently to another agent
6. **Reflection exercise** — tool descriptions matter more than you think

### Core SDK primitives

| Primitive | Description |
|---|---|
| `Agent` | An LLM with instructions, tools, and optional handoffs |
| `Runner` | Executes an agent and manages the agentic loop |
| `function_tool` | Decorator that exposes a Python function as an agent tool |
| `handoff` | One-way transfer of control to another agent |

In [ ]:
import os
import math
import numpy as np
import faiss
from datetime import date
from openai import OpenAI
from agents import Agent, Runner, function_tool, handoff
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
print("Setup complete — OpenAI Agents SDK ready")

## 1. Hello Agent — The Simplest Case

An `Agent` is an LLM with instructions. `Runner.run_sync()` runs it and
manages the loop until the agent produces a final output.

In [ ]:
# The simplest possible agent: no tools, just instructions + model
hello_agent = Agent(
    name="Course Assistant",
    instructions=(
        "You are a helpful assistant for an applied NLP course. "
        "Be concise — answer in 2–3 sentences."
    ),
    model="gpt-4o-mini",
)

result = Runner.run_sync(hello_agent, "What is the difference between a workflow and an agent?")
print(result.final_output)

### What does `result` contain?

In [ ]:
print(f"Type:          {type(result).__name__}")
print(f"Last agent:    {result.last_agent.name}")
print(f"Steps taken:   {len(result.new_items)}")
print()
for item in result.new_items:
    print(f"  {item.type}")

## 2. Adding Tools

Tools are Python functions decorated with `@function_tool`.
The docstring becomes the tool description that the agent reads to decide
when and how to use the tool.

### Structured output in the Agents SDK — two places it appears

The `BaseModel` pattern from Session 8 appears in two distinct places in the SDK:

**1. Tool return types** — a tool function returns a Pydantic model instead of a string.
The agent receives a typed, structured result it can rely on, rather than text it has to interpret.

**2. `output_type` on `Agent`** — constrains the agent's *final answer* to a Pydantic schema.
```python
agent = Agent(output_type=MyModel, ...)
result = Runner.run_sync(agent, "question")
result.final_output   # MyModel instance — not a string
```
The SDK passes the schema to the LLM and validates the response.
`result.final_output` is then a typed Python object — fields are directly accessible,
no parsing required. The agent can still call tools freely; only the final answer must match the schema.

We use both in this notebook:
- `calculate` returns a `CalcResult` model (tool return type)
- `researcher` agent uses `output_type=ResearchSummary` (agent output type — shown later)

We define three tools:
- `calculate` — returns a `CalcResult` Pydantic model (structured output)
- `get_todays_date` — returns a plain string (simple enough to not need a model)
- `search_course_materials` — returns formatted string passages

In [ ]:
from pydantic import BaseModel

class CalcResult(BaseModel):
    expression: str
    result: float
    formatted: str  # human-readable: "0.85 ** 10 = 0.1969"

@function_tool
def calculate(expression: str) -> CalcResult:
    """
    Evaluate a mathematical expression safely.
    Supports: +, -, *, /, **, sqrt, log, sin, cos, pi, e.
    Examples: '0.85 ** 10', 'math.sqrt(144)', '(3 + 4) * 2'
    Returns a structured result with the expression, numeric value, and formatted string.
    """
    safe_env = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    safe_env["__builtins__"] = {}
    try:
        value = float(eval(expression, safe_env))
        return CalcResult(
            expression=expression,
            result=value,
            formatted=f"{expression} = {value}",
        )
    except Exception as e:
        return CalcResult(expression=expression, result=0.0, formatted=f"Error: {e}")

@function_tool
def get_todays_date() -> str:
    """Return today's date in YYYY-MM-DD format."""
    return str(date.today())

# Test tools directly before giving them to an agent
result_calc = calculate("0.85 ** 10")
print(type(result_calc))          # CalcResult — a typed object, not a string
print(result_calc.result)         # 0.1968...  — directly accessible field
print(result_calc.formatted)      # "0.85 ** 10 = 0.1968..."
print(get_todays_date())

In [ ]:
# Build a small in-memory FAISS index over course content
# (Same texts as notebook 01 — in a real app you'd load this from disk)
COURSE_TEXTS = [
    "Retrieval-Augmented Generation (RAG) combines a retrieval system with an LLM to ground answers in documents.",
    "FAISS is a library for efficient similarity search over dense vectors, built by Meta.",
    "Cosine similarity measures the angle between two vectors, returning a value between -1 and 1.",
    "An embedding is a dense numerical representation of text that captures semantic meaning.",
    "BM25 is a lexical search algorithm scoring by term frequency and inverse document frequency.",
    "The OpenAI Agents SDK uses Agent, Runner, function_tool, and handoffs as its core primitives.",
    "Agentic RAG lets an LLM iteratively retrieve and refine its answer rather than retrieving once.",
    "MCP (Model Context Protocol) is an open standard for connecting AI models to tools and data sources.",
    "Prompt chaining passes the output of one LLM call as the input to the next.",
    "The compound accuracy problem: a 10-step workflow at 85% accuracy per step succeeds only 20% of the time.",
]

def _embed(texts: list[str]) -> np.ndarray:
    response = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return np.array([item.embedding for item in response.data], dtype=np.float32)

_course_vecs = _embed(COURSE_TEXTS)
_course_index = faiss.IndexFlatL2(_course_vecs.shape[1])
_course_index.add(_course_vecs)
print(f"Course index ready: {_course_index.ntotal} vectors")

@function_tool
def search_course_materials(query: str) -> str:
    """
    Search ANLP course materials for information relevant to the query.
    Returns the 2 most relevant passages.
    Use this for questions about course concepts, terminology, and techniques.
    """
    q_vec = _embed([query]).reshape(1, -1)
    _, indices = _course_index.search(q_vec, 2)
    passages = [COURSE_TEXTS[i] for i in indices[0]]
    return "\n\n".join(f"[{i+1}] {p}" for i, p in enumerate(passages))

# Test the search tool
print(search_course_materials("how does semantic search work?"))

### Agent with all three tools

The agent now decides which tools to call — and in what order — based on the question.

In [ ]:
tool_agent = Agent(
    name="Course Assistant",
    instructions="""You are a helpful assistant for an applied NLP course.
    - Use search_course_materials for questions about course content and concepts
    - Use calculate for any mathematical computation
    - Use get_todays_date when the current date is relevant
    Always explain your reasoning and which tools you used.""",
    tools=[search_course_materials, calculate, get_todays_date],
    model="gpt-4o-mini",
)

result = Runner.run_sync(
    tool_agent,
    "What is cosine similarity, and if each step in a 10-step pipeline is 85% accurate, "
    "what is the overall success rate?",
)
print(result.final_output)

## 3. True Agentic RAG — The LLM Controls the Loop

In notebook 01 we built a manual RAG loop: the programmer decided when to retrieve,
when to check completeness, when to rewrite the query.

Now we give the LLM a `retrieve` tool and let `Runner` manage the loop.
The LLM decides:
- **if** it needs to retrieve
- **what** query to use
- **how many times** to retrieve
- **when** it has enough context to answer

Same outcome. Different control. This is the difference between a workflow and an agent.

In [ ]:
@function_tool
def retrieve(query: str) -> str:
    """
    Search ANLP course materials for information relevant to the query.
    Returns the 3 most relevant passages.
    Call this whenever you need factual grounding before answering.
    You decide when to call it, what to search for, and whether to call it again.
    """
    q_vec = _embed([query]).reshape(1, -1)
    _, indices = _course_index.search(q_vec, 3)
    passages = [COURSE_TEXTS[i] for i in indices[0]]
    return "\n\n".join(f"[{i+1}] {p}" for i, p in enumerate(passages))

rag_agent = Agent(
    name="RAG Agent",
    instructions=(
        "Answer questions by searching course materials first. "
        "Search as many times as needed with different queries until you have enough information. "
        "Only produce a final answer when you are confident it is complete and accurate."
    ),
    tools=[retrieve],
    model="gpt-4o-mini",
)

rag_result = Runner.run_sync(
    rag_agent,
    "How does FAISS support semantic search and what similarity metric does it use?",
)
print(rag_result.final_output)

### What did the agent decide?

Inspect the trace to see the LLM's decisions — queries it chose, number of retrievals,
when it decided it had enough context.

In [ ]:
print(f"Steps the agent took: {len(rag_result.new_items)}\n")

for i, item in enumerate(rag_result.new_items, 1):
    if item.type == "tool_call_item":
        import json
        args = json.loads(item.raw_item.arguments)
        print(f"Step {i} — RETRIEVE")
        print(f"  Query the LLM chose: {args.get('query', '')}")
    elif item.type == "tool_call_output_item":
        output = str(item.raw_item.get("output", ""))
        print(f"Step {i} — Retrieved {len(output)} chars of context")
    elif item.type == "message_output_item":
        print(f"Step {i} — LLM decided it had enough → produced final answer")
    print()

### Workflow vs. Agent — Side by Side

| | Manual `agentic_rag` (notebook 01) | `rag_agent` (this notebook) |
|---|---|---|
| Who decides to retrieve? | Programmer's `for` loop | LLM, via tool call |
| Who decides when to stop? | Programmer's `is_complete()` | LLM producing final output |
| Who decides what to search? | Programmer's `rewrite_query()` | LLM choosing its own query |
| Max iterations enforced by | `max_iterations` parameter | `Runner` (default: 10 turns) |
| Control flow lives in | Your Python code | The model's reasoning |

The agent's queries are different from ours. Its number of retrievals varies by question.
It stops when *it* is satisfied — not when our code tells it to stop.

This is what "model-directed control flow" means in practice.

## 4. Inspecting the Trace

Two ways to inspect what an agent did:

**Local inspection** — `result.new_items` in the notebook. Immediate, no setup.
Good for step-by-step teaching and quick debugging.

**OpenAI Platform tracing** — sends the run to `platform.openai.com/traces`.
Persistent, visual, shareable. What you'd use in production.

We cover both.

In [ ]:
print(f"Total steps: {len(result.new_items)}\n")

for i, item in enumerate(result.new_items, 1):
    if item.type == "tool_call_item":
        print(f"Step {i} — TOOL CALL")
        print(f"  Tool:  {item.raw_item.name}")
        print(f"  Input: {item.raw_item.arguments}")

    elif item.type == "tool_call_output_item":
        output_str = str(item.raw_item.get("output", ""))
        preview = output_str[:100] + ("..." if len(output_str) > 100 else "")
        print(f"Step {i} — TOOL RESULT")
        print(f"  Output: {preview}")

    elif item.type == "message_output_item":
        text = item.raw_item.content[0].text if item.raw_item.content else ""
        print(f"Step {i} — FINAL MESSAGE ({len(text)} chars)")
        print(f"  Preview: {text[:80]}...")

    print()

### OpenAI Platform Tracing

The `trace` context manager sends the run to the OpenAI Platform traces UI.
You can inspect the full workflow visually, share the URL with a colleague,
and track token usage and latency — all without adding any logging code.

Three additions to a normal `Runner.run_sync()` call:
1. Wrap with `trace(workflow_name="...")` — names the trace in the UI
2. Pass `RunConfig(trace_metadata={...})` — attaches searchable labels
3. Print the trace URL — a direct link to view it in the platform

In [ ]:
from agents import trace, RunConfig
from datetime import datetime

# Name the workflow — appears as the title in platform.openai.com/traces
workflow_name = f"session9_demo_{datetime.now().strftime('%H%M%S')}"

with trace(workflow_name=workflow_name):
    traced_result = Runner.run_sync(
        tool_agent,
        "What is cosine similarity, and what is 0.85 to the power of 10?",
        run_config=RunConfig(
            trace_metadata={"session": "9", "demo": "platform_tracing"},
        ),
    )

print(traced_result.final_output)
print(f"\nView trace at: https://platform.openai.com/traces")

### Trace continuity — grouping multiple runs

Use the same `trace_id` across multiple `Runner.run_sync()` calls to group them
as one workflow session in the UI. Useful for multi-turn conversations or
multi-step pipelines where you want to see everything in one view.

In [ ]:
from agents import trace, RunConfig

# Start a named trace manually — gives us the trace_id to reuse
session_trace = trace(
    workflow_name="session9_multi_turn",
    group_id="anlp_session9",
    metadata={"lecturer": "Mutaz", "cohort": "AUT2026"},
)
session_trace.start()
trace_id = session_trace.trace_id
print(f"Trace ID: {trace_id}")

# Shared RunConfig — all runs share the same trace_id
run_cfg = RunConfig(
    trace_id=trace_id,
    workflow_name="session9_multi_turn",
    group_id="anlp_session9",
    trace_metadata={"agent": "Course Assistant"},
)

# Three separate runs — all appear grouped under one trace in the UI
questions = [
    "What is a vector embedding?",
    "What is 0.85 to the power of 10?",
    "What does FAISS stand for?",
]

for q in questions:
    r = Runner.run_sync(tool_agent, q, run_config=run_cfg)
    print(f"Q: {q}")
    print(f"A: {r.final_output[:100]}...\n")

session_trace.finish()

trace_url = f"https://platform.openai.com/traces/trace?trace_id={trace_id}"
print(f"Full trace (all 3 runs grouped): {trace_url}")

## 5. Orchestrator-Workers — Agents as Tools

In the orchestrator-workers pattern:
- A **central orchestrator** receives the task
- It delegates to **specialist workers** using `agent.as_tool()`
- The orchestrator **retains control** — it sees the worker's output and decides what to do next

This is different from a handoff (which permanently transfers control).

### `output_type` — structured output at the agent level

Recall from the tool section: structured output can appear at two places in the SDK.
Here we use the second: `output_type` on the agent itself.

#### Important: Pydantic objects are a Python-layer concept — the LLM always sees JSON

The LLM never receives a Python object. Every boundary between the LLM and the outside
world is serialised text. Here is what actually crosses each boundary:

| Boundary | What crosses it |
|---|---|
| Tool → LLM (tool result) | Serialised JSON string of the Pydantic model |
| LLM → Python caller (`result.final_output`) | SDK parses LLM's JSON → Python object |
| Worker-as-tool → Orchestrator LLM | Serialised JSON string of the worker's output |
| Orchestrator LLM → Python caller | String (unless orchestrator also has `output_type`) |

#### What `output_type` gives you

When running an agent **standalone**, `output_type` gives the Python caller a typed object:

```python
researcher = Agent(output_type=ResearchSummary, ...)
result = Runner.run_sync(researcher, "question")

# Python caller gets a typed object — not a string
result.final_output.key_findings   # list[str], directly accessible
result.final_output.topic          # str
```

**Summary:** `output_type` gives the **Python programmer** typed access to final outputs.
It gives the **LLM** predictable, well-structured JSON to reason over.

In [ ]:
from pydantic import BaseModel

class ResearchSummary(BaseModel):
    topic: str
    key_findings: list[str]   # bullet-point facts found in the course materials
    passages_used: list[str]  # the retrieved passages the findings are based on

researcher = Agent(
    name="Researcher",
    instructions="""You are a research specialist for an NLP course.
    Search the course materials thoroughly.
    Return your findings as a ResearchSummary with:
    - topic: the subject you researched
    - key_findings: 2–4 bullet-point facts from the materials
    - passages_used: the exact retrieved passages you used""",
    tools=[search_course_materials],
    output_type=ResearchSummary,
    model="gpt-4o-mini",
)

explainer = Agent(
    name="Explainer",
    instructions="""You are a teaching specialist.
    Take technical content and explain it clearly for Master's students.
    Use one concrete analogy. Avoid unnecessary jargon.""",
    model="gpt-4o-mini",
)

print("Worker agents defined:")
print(f"  {researcher.name} — output_type=ResearchSummary (structured)")
print(f"  {explainer.name} — output_type=str (free text)")

In [ ]:
# Run the researcher on its own to see the structured output before wiring it into the orchestrator
research_result = Runner.run_sync(
    researcher,
    "What is the compound accuracy problem?",
)

# final_output is a ResearchSummary object — not a string
summary: ResearchSummary = research_result.final_output
print(f"Type: {type(summary).__name__}")
print(f"Topic: {summary.topic}")
print(f"Key findings ({len(summary.key_findings)}):")
for finding in summary.key_findings:
    print(f"  • {finding}")
print(f"Passages used: {len(summary.passages_used)}")

In [ ]:
orchestrator = Agent(
    name="Orchestrator",
    instructions="""You coordinate research and explanation tasks.
    For any question:
    1. Use 'research' to find relevant course material
    2. Use 'explain' to make the findings clear for students
    3. Combine into a well-structured final answer""",
    tools=[
        researcher.as_tool(
            tool_name="research",
            tool_description=(
                "Search ANLP course materials and return a ResearchSummary with "
                "key_findings and passages_used. Use for factual lookups about course content."
            ),
        ),
        explainer.as_tool(
            tool_name="explain",
            tool_description=(
                "Take technical content and rephrase it clearly for Master's students. "
                "Use after research to make findings student-friendly."
            ),
        ),
    ],
    model="gpt-4o-mini",
)

orch_result = Runner.run_sync(
    orchestrator,
    "Explain the compound accuracy problem in multi-step agentic pipelines",
)
print(orch_result.final_output)

In [ ]:
# draw_graph requires: pip install graphviz AND Graphviz system binary (graphviz.org/download)
# Without the system binary the try/except prints a text fallback instead.
try:
    from agents.extensions.visualization import draw_graph
    draw_graph(orchestrator)  # renders inline as SVG in Jupyter
except ImportError:
    print("graphviz Python package not installed — run: pip install graphviz")
except Exception:
    # System binary missing — show text diagram instead
    print("""
Orchestrator architecture (text fallback):

  __start__
      │
  Orchestrator
  ├──⟶ research   (Researcher.as_tool — searches course materials, returns ResearchSummary JSON)
  └──⟶ explain    (Explainer.as_tool — rewrites findings for students)
      │
   __end__

Dotted edges = tool calls (agent retains control after each call)
""")

## 6. Handoffs — Permanently Transferring Control

A **handoff** is a one-way transfer: the routing agent hands off to a specialist
and is done. The specialist owns the rest of the conversation.

Compare with agents-as-tools:
- **Agent as tool**: orchestrator calls worker, gets result, continues
- **Handoff**: router transfers control to specialist — router is done

`result.last_agent` tells you who finished the conversation.

In [ ]:
billing_agent = Agent(
    name="Billing Specialist",
    instructions=(
        "You handle billing and payment questions. "
        "Be precise, acknowledge the issue clearly, and offer concrete next steps."
    ),
    model="gpt-4o-mini",
)

technical_agent = Agent(
    name="Technical Support",
    instructions=(
        "You handle technical issues, bugs, and errors. "
        "Ask one clarifying question if needed to diagnose the issue."
    ),
    model="gpt-4o-mini",
)

router = Agent(
    name="Triage Router",
    instructions=(
        "Route queries to the right specialist immediately. Do not answer yourself.\n"
        "- Billing, invoice, payment, charge → Billing Specialist\n"
        "- Technical problems, bugs, crashes, errors → Technical Support"
    ),
    handoffs=[
        handoff(billing_agent),
        handoff(technical_agent),
    ],
    model="gpt-4o-mini",
)

queries = [
    "I was charged twice for my subscription this month",
    "The app crashes every time I try to export a PDF",
]

for q in queries:
    r = Runner.run_sync(router, q)
    print(f"Query:       {q}")
    print(f"Handled by:  {r.last_agent.name}")
    print(f"Response:    {r.final_output[:120]}...")
    print()

In [ ]:
# Handoffs produce the clearest draw_graph output:
# solid directed arrows show which agents the router can hand off to.
try:
    from agents.extensions.visualization import draw_graph
    draw_graph(router)  # renders inline as SVG in Jupyter
except ImportError:
    print("graphviz Python package not installed — run: pip install graphviz")
except Exception:
    print("""
Handoff architecture (text fallback):

  __start__
      │
  Triage Router
  ├──▶ Billing Specialist    [handoff — router done, specialist takes over]
  └──▶ Technical Support     [handoff — router done, specialist takes over]

Solid arrows = handoffs (control transfers permanently to the specialist)
Compare with orchestrator-workers above: dotted arrows = agent retains control
""")

### Agents as Tools vs. Handoffs — Side by Side

```
AGENTS AS TOOLS                      HANDOFFS
──────────────────────────────────   ──────────────────────────────────
Orchestrator calls worker            Router gives away control
Orchestrator sees the result         Router is done after the handoff
Can call multiple workers in one run One-way — specialist takes over
result.last_agent = Orchestrator     result.last_agent = Specialist

Use when: you need the sub-result    Use when: the specialist should
as input for the next step           own the rest of the conversation
```

## 7. Reflection: Tool Descriptions Matter

The tool's docstring is part of the agent's prompt.
A vague description leads to unpredictable tool use.
Watch what happens when we give the agent a tool called `data` with no useful description.

In [ ]:
# Shared underlying implementation — both tools call the same logic
def _do_search(query: str) -> str:
    q_vec = _embed([query]).reshape(1, -1)
    _, indices = _course_index.search(q_vec, 2)
    return "\n\n".join(f"[{i+1}] {COURSE_TEXTS[j]}" for i, j in enumerate(indices[0]))

In [ ]:
@function_tool
def data(q: str) -> str:
    """Get data."""  # ← no context: what data? for what? when to use it?
    return _do_search(q)

vague_agent = Agent(
    name="Vague Agent",
    instructions="Answer questions about the course.",
    tools=[data],
    model="gpt-4o-mini",
)

r_vague = Runner.run_sync(vague_agent, "What is MCP and why does it matter?")
tool_calls_vague = sum(1 for item in r_vague.new_items if item.type == "tool_call_item")
print(f"=== Vague tool description ===")
print(f"Tool calls made: {tool_calls_vague}")
print(f"Answer preview:  {r_vague.final_output[:200]}")

In [ ]:
@function_tool
def search_course_materials_v2(query: str) -> str:
    """
    Search ANLP course materials for information relevant to a query.
    Returns the 2 most relevant passages from lecture notes and reference material.
    Use this for questions about course concepts, terminology, and techniques.
    Do NOT use for mathematical calculations — use calculate for that.
    """
    return _do_search(query)

good_agent = Agent(
    name="Good Agent",
    instructions="Answer questions about the course.",
    tools=[search_course_materials_v2],
    model="gpt-4o-mini",
)

r_good = Runner.run_sync(good_agent, "What is MCP and why does it matter?")
tool_calls_good = sum(1 for item in r_good.new_items if item.type == "tool_call_item")
print(f"=== Good tool description ===")
print(f"Tool calls made: {tool_calls_good}")
print(f"Answer preview:  {r_good.final_output[:200]}")

print(f"""
─────────────────────────────────────
💡 Lesson: The docstring IS the interface.
   Vague descriptions → agents skip tools or misuse them
   Clear descriptions → reliable, predictable tool use

   The tool description deserves as much care as the tool implementation.
""")

## Summary

| Pattern | Key idea | SDK primitive |
|---|---|---|
| Basic agent | LLM + instructions + loop | `Agent`, `Runner.run_sync()` |
| Tool use | Agent decides when to call Python functions | `@function_tool` |
| Agentic RAG | LLM controls its own retrieval loop | `@function_tool` + `Agent` |
| Trace inspection | See every step the agent took | `result.new_items` |
| Platform tracing | Visual, persistent trace in the cloud | `trace()`, `RunConfig` |
| Orchestrator-workers | Orchestrator delegates; retains control | `agent.as_tool()` |
| Handoffs | Router transfers control permanently | `handoff(agent)` |
| Tool descriptions | Docstring = agent's interface | Write clearly |

### Further reading
- OpenAI Agents SDK docs: https://openai.github.io/openai-agents-python/
- Anthropic "Building Effective Agents": https://www.anthropic.com/engineering/building-effective-agents
- OpenAI "A Practical Guide to Building Agents": https://cdn.openai.com/business-guides-and-resources/a-practical-guide-to-building-agents.pdf